# Xarray-Spatial Terrain Metrics: Landform classification

The Weiss (2001) landform scheme assigns each raster cell to one of ten terrain categories based on its topographic position at two scales plus its slope. It takes raw elevation and produces a categorical map that separates canyons, valleys, plains, slopes, ridges, and mountaintops. Useful for habitat mapping, geomorphology, and land-use planning where you need terrain types rather than raw numbers.

### What you'll build

1. Generate synthetic terrain with ridges, valleys, and rolling hills
2. Run the ten-class landform classifier with default parameters
3. Explore how neighborhood radii change the classification scale
4. Adjust the slope threshold that separates plains from open slopes
5. Examine the distribution of landform classes across the raster

![Landform classification preview](images/landform_classification_preview.png)

[Terrain data](#Terrain-data) · [Landform classification](#Landform-classification) · [Neighborhood radii](#Neighborhood-radii) · [Slope threshold](#Slope-threshold) · [Class distribution](#Class-distribution)

Standard imports plus `landforms` and the class-name lookup dictionary.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

import xrspatial
from xrspatial import landforms
from xrspatial.terrain_metrics import LANDFORM_CLASSES

## Terrain data

Synthetic elevation generated with `generate_terrain`. The same raster is reused in every section below.

In [ ]:
W = 800
H = 600
x_range = (-20e6, 20e6)
y_range = (-20e6, 20e6)

terrain = xr.DataArray(np.zeros((H, W)))
terrain = terrain.xrs.generate_terrain(x_range=x_range, y_range=y_range)
illuminated = terrain.xrs.hillshade()

terrain.plot.imshow(cmap='terrain', size=7.5, aspect=W/H, add_colorbar=False)

Low areas are blue-green, ridges are brown and white. Plenty of topographic variety for the landform classifier to work with.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
terrain.plot.imshow(ax=ax, cmap='terrain', alpha=128/255, add_colorbar=False)
ax.set_axis_off()

## Landform classification

`landforms()` implements the [Weiss (2001) TPI-based classification](http://www.jennessent.com/downloads/tpi-poster-tnc_18x22.pdf). It computes Topographic Position Index at two neighborhood scales, standardizes both to z-scores, and combines them with slope to sort each cell into one of ten categories. The `inner_radius` and `outer_radius` parameters control the two TPI scales in cells.

The plot shows all ten classes overlaid on hillshade, with a legend mapping colors to landform names.

In [ ]:
# Run the classifier
classes = landforms(terrain, inner_radius=3, outer_radius=15)

# Ten-class colormap (colorblind-safe: no red/green pairing)
landform_colors = [
    '#1a237e',  # 1  Canyon
    '#4a148c',  # 2  Midslope drainage
    '#880e4f',  # 3  Upland drainage
    '#0d47a1',  # 4  U-shaped valley
    '#a5d6a7',  # 5  Plain
    '#fff9c4',  # 6  Open slope
    '#ffcc80',  # 7  Upper slope
    '#e65100',  # 8  Local ridge
    '#bf360c',  # 9  Midslope ridge
    '#b71c1c',  # 10 Mountain top
]
cmap = ListedColormap(landform_colors)
bounds = np.arange(0.5, 11.5, 1)
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(10, 7.5))
illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
classes.plot.imshow(ax=ax, cmap=cmap, norm=norm, alpha=0.5, add_colorbar=False)
ax.legend(
    handles=[Patch(facecolor=landform_colors[i], alpha=0.78,
                   label=LANDFORM_CLASSES[i + 1])
             for i in range(10)],
    loc='lower right', fontsize=8, framealpha=0.9, ncol=2,
)
ax.set_axis_off()

Dark blues and purples sit in the valleys and drainages, greens and yellows cover the flats and gentle slopes, and oranges and reds mark the ridges and peaks. The spatial pattern lines up with what the hillshade shows.

### Isolating individual landforms

A pop-art style breakdown of four representative classes spanning the full topographic range: canyons at the bottom, plains in the flats, local ridges on the shoulders, and mountain tops at the peaks. Each panel masks a single class in neon over the muted hillshade so you can see exactly where that landform type sits on the terrain.

In [ ]:
# Warhol-style facet: four landforms isolated in neon on muted terrain
picks = [
    (1,  'Canyon',       '#ff2a6d'),   # hot pink
    (5,  'Plain',        '#05d9e8'),   # electric cyan
    (8,  'Local ridge',  '#39ff14'),   # neon green
    (10, 'Mountain top', '#ff6e27'),   # neon orange
]

fig, axes = plt.subplots(2, 2, figsize=(13, 10), facecolor='#1a1a1a')
fig.subplots_adjust(wspace=0.04, hspace=0.12)

for ax, (cls_id, name, neon) in zip(axes.flat, picks):
    ax.set_facecolor('#2a2a2a')
    # Muted hillshade backdrop
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False,
                            alpha=0.4)
    # Isolate a single landform class
    mask = xr.where(classes == cls_id, 1.0, np.nan)
    mask.plot.imshow(ax=ax, cmap=ListedColormap([neon]),
                     vmin=0.5, vmax=1.5, alpha=0.85,
                     add_colorbar=False)
    ax.set_title(name.upper(), color=neon, fontsize=15,
                 fontweight='bold', fontfamily='monospace', pad=8)
    ax.set_axis_off()

<div class="alert alert-block alert-warning">
<b>Radii are in cells, not meters.</b> The <code>inner_radius</code> and <code>outer_radius</code> parameters count grid cells, so the actual ground distance depends on your raster's cell size. A 15-cell radius means 450 m on a 30 m DEM but only 15 m on a 1 m lidar raster. Pick radii that match the scale of the landforms you care about.
</div>

## Neighborhood radii

The inner and outer radii control the scale of features the classifier picks up. Smaller radii detect fine-grained features like narrow gullies. Larger radii smooth out local variation and respond to broader landforms. Both radii need to be tuned to the resolution and landscape scale of your data.

Three panels show the same terrain classified with progressively wider neighborhoods.

In [ ]:
configs = [
    (2, 6, 'inner=2, outer=6'),
    (3, 15, 'inner=3, outer=15'),
    (5, 25, 'inner=5, outer=25'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (ir, outr, label) in zip(axes, configs):
    c = landforms(terrain, inner_radius=ir, outer_radius=outr)
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    c.plot.imshow(ax=ax, cmap=cmap, norm=norm, alpha=0.5, add_colorbar=False)
    ax.set_title(label, fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

The narrow radii (left panel) produce a noisy, fragmented map because TPI at small scales responds to individual bumps. The wide radii (right panel) smooth everything into broad zones. The middle setting balances detail and coherence for this terrain.

## Slope threshold

The `slope_threshold` parameter (default 5 degrees) decides whether cells in the middle TPI range get labeled as plains (class 5) or open slopes (class 6). Cells with both small-scale and large-scale TPI near zero are in a neutral topographic position. Slope breaks the tie: flat cells become plains, tilted cells become open slopes.

A lower threshold classifies more of the surface as slopes; a higher one expands the plain category. The three panels below show the effect.

In [ ]:
thresholds = [2.0, 5.0, 15.0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, thr in zip(axes, thresholds):
    c = landforms(terrain, inner_radius=3, outer_radius=15,
                  slope_threshold=thr)
    illuminated.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    c.plot.imshow(ax=ax, cmap=cmap, norm=norm, alpha=0.5, add_colorbar=False)
    ax.set_title(f'slope_threshold={thr}', fontsize=13)
    ax.set_axis_off()
plt.tight_layout()

At threshold=2 (left), nearly all mid-position cells become open slopes. At threshold=15 (right), most of them flip to plains. The default of 5 degrees is a reasonable starting point, but the right value depends on the relief of your study area.

<div class="alert alert-block alert-info">
<b>Slope units.</b> The <code>slope_threshold</code> is in degrees. The internal <code>slope()</code> call inherits its units from the elevation array's coordinate spacing. For projected data with matching horizontal and vertical units this works as expected. For geographic (lat/lon) data, use <code>xrspatial.slope(agg, use_great_circle='GREAT_CIRCLE')</code> separately and verify units before relying on the threshold.
</div>

## Class distribution

A horizontal bar chart of class frequencies shows which landform types dominate the raster. This is a quick sanity check: if one class swallows everything, the radii or threshold probably need tuning.

In [ ]:
valid = classes.values[~np.isnan(classes.values)].astype(int)
counts = [np.sum(valid == i) for i in range(1, 11)]
labels = [LANDFORM_CLASSES[i] for i in range(1, 11)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(10), counts, color=landform_colors)
ax.set_yticks(range(10))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Cell count')
ax.invert_yaxis()
plt.tight_layout()

### References

- [Weiss, A. (2001). Topographic Position and Landforms Analysis](http://www.jennessent.com/downloads/tpi-poster-tnc_18x22.pdf), ESRI International User Conference poster
- [Topographic Position Index overview](https://en.wikipedia.org/wiki/Topographic_Position_Index), Wikipedia
- [xrspatial terrain_metrics API docs](https://makepath.github.io/xarray-spatial/reference/_autosummary/xrspatial.terrain_metrics.html)